In [1]:
import pandas as pd

In [2]:
df = pd.DataFrame(pd.read_csv("/mnt/common-ssd/karakasvioletta/valasztokeruletek/data/processed/comparison/egyszerusitett_egyezesek.csv"))

In [3]:
print(df.head(1))

   település_azonosító közterület_neve_22 közterület_jellege_22 házszám_22  \
0                 1001             Mihály                  utca     000016   

  épület_22 lépcsőház_22  szavazókör_22  OEVK_22 közterület_neve_norm  \
0         A          NaN             20        1               mihaly   

  közterület_jellege_norm_22  ... épület_26 lépcsőház_26 szavazókör_26  \
0                       utca  ...       NaN          NaN            16   

   OEVK_26 közterület_jellege_norm_26 épület_norm_26 lépcsőház_norm_26  \
0        1                       utca            NaN               NaN   

  _record_id_26     egyeztetesi_modszer  kulonbsegek  
0             4  egyszerusitett_egyezes       épület  

[1 rows x 27 columns]


In [4]:
print(df["kulonbsegek"].value_counts())

kulonbsegek
közterület_jellege               1177
épület                            676
lépcsőház                           8
közterület_jellege; épület          7
épület; lépcsőház                   6
közterület_jellege; lépcsőház       5
Name: count, dtype: int64


In [5]:
kulonbsegek = (
    df["kulonbsegek"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
)

# Az egyes különbségek megszámolása
kulonbseg_db = kulonbsegek.value_counts()

print(kulonbseg_db)

kulonbsegek
közterület_jellege    1189
épület                 689
lépcsőház               19
Name: count, dtype: int64


közterület jellegű különbségek

In [6]:
# Csak azok a sorok, ahol a közterület jellege eltér
jelleg_elteresek = df[
    df["kulonbsegek"].str.contains(
        "közterület_jellege",
        na=False
    )
].copy()

# Csoportosítás és darabszámolás
jelleg_osszesites = (
    jelleg_elteresek
    .groupby([
        "település_azonosító",
        "közterület_neve_22",
        "közterület_jellege_22",
        "közterület_jellege_26"
    ])
    .size()
    .reset_index(name="darab")
    .sort_values("darab", ascending=False)
)

print(jelleg_osszesites)
print((jelleg_osszesites["darab"] == 1).sum())
print((jelleg_osszesites["darab"] < 5).sum())

     település_azonosító közterület_neve_22 közterület_jellege_22  \
65                 11046     Rákóczi Ferenc                    út   
29                  5166      Kossuth Lajos                  utca   
67                 11052             Temető                  utca   
63                 10085          Szabadság                    út   
104                19010            Somhegy                puszta   
..                   ...                ...                   ...   
103                16227              Árpád                    út   
106                19119             Orgona                   köz   
105                19027             Zombor                   köz   
108                20150          Olgamajor             erdészház   
110                20231            Kápolna                  utca   

    közterület_jellege_26  darab  
65                   utca     82  
29                     út     75  
67                     út     39  
63                   utca     3

In [24]:
print(
    df[
        (df["kulonbsegek"] == "közterület_jellege; épület")
        | (df["kulonbsegek"] == "közterület_jellege; lépcsőház")
    ][[
        "település_azonosító",
        "közterület_neve_22",
        "közterület_jellege_22",
        "közterület_jellege_26",
        "házszám_22",
        "épület_22",
        "épület_26",
        "lépcsőház_22",
        "lépcsőház_26"
    ]].to_string(index=False)
)

print(len(df[
        (df["kulonbsegek"] == "közterület_jellege; épület")
        | (df["kulonbsegek"] == "közterület_jellege; lépcsőház")]))

 település_azonosító közterület_neve_22 közterület_jellege_22 közterület_jellege_26 házszám_22 épület_22 épület_26 lépcsőház_22 lépcsőház_26
                1003        Farkastorki                 lejtő                    út     000004       NaN         E          NaN          NaN
                4006           Stefánia                  utca                   sor     000003      0001       NaN          NaN          NaN
                8104                Vár                   NaN                   tér     000002       NaN         F          NaN          NaN
                9009       Dózsa György                  utca                 udvar     000005       NaN       NaN         0001          NaN
               14004            Templom                  utca                  kert     000007       NaN         I          NaN          NaN
               14015             Otthon                  utca                   köz     000005         B       NaN          NaN          NaN
             

épület

In [18]:
epulet_elteresek = df[
    df["kulonbsegek"].str.contains(
        "épület",
        na=False
    )
].copy()

# Hány olyan eltérés van, ahol az egyik épületadat hiányzik?
nan_eltérések = (
    epulet_elteresek["épület_22"].isna()
    | epulet_elteresek["épület_26"].isna()
).sum()

print("Épületeltérések, ahol az egyik adat NaN:", nan_eltérések)

# Csoportosítás és darabszámolás
epulet_osszesites = (
    epulet_elteresek
    .groupby([
        "település_azonosító",
        "közterület_neve_22",
        "közterület_jellege_22",
        "közterület_jellege_26",
        "épület_22",
        "épület_26"
    ], dropna=False)
    .size()
    .reset_index(name="darab")
    .sort_values("darab", ascending=False)
)

print(epulet_osszesites.head(5).to_string(index=False))

print(
    "\nOlyan csoportok száma, ahol a darab = 1:",
    (epulet_osszesites["darab"] == 1).sum()
)

mindketto_van = epulet_elteresek[
    epulet_elteresek["épület_22"].notna()
    & epulet_elteresek["épület_26"].notna()
].copy()

print("Mindkét évben van épületadat:", len(mindketto_van))

print(mindketto_van[[
    "település_azonosító",
    "közterület_neve_22",
    "közterület_jellege_22",
    "házszám_22",
    "épület_22",
    "épület_26"
]].to_string(index=False))

Épületeltérések, ahol az egyik adat NaN: 678
 település_azonosító közterület_neve_22 közterület_jellege_22 közterület_jellege_26 épület_22 épület_26  darab
                2144      Hunyadi János                  utca                  utca         E       NaN     12
               14147              Kátai                    út                    út       NaN         A      8
                4067           Vásártér                  utca                  utca       NaN         A      6
                9053               Kfcs                  part                  part       NaN         H      4
                3056  Vörösmarty Mihály                  utca                  utca       NaN         A      4

Olyan csoportok száma, ahol a darab = 1: 584
Mindkét évben van épületadat: 11
 település_azonosító közterület_neve_22 közterület_jellege_22  házszám_22 épület_22 épület_26
                1014            Thököly                    út      000127      0001         A
                2089  

lépcsőház

In [19]:
lh_elteresek = df[
    df["kulonbsegek"].str.contains(
        "lépcsőház",
        na=False
    )
].copy()

# Hány olyan eltérés van, ahol az egyik épületadat hiányzik?
nan_eltérések_lh = (
    lh_elteresek["épület_22"].isna()
    | lh_elteresek["épület_26"].isna()
).sum()

print("Lépcsőházeltérések, ahol az egyik adat NaN:", nan_eltérések_lh)

# Csoportosítás és darabszámolás
lh_osszesites = (
    lh_elteresek
    .groupby([
        "település_azonosító",
        "közterület_neve_22",
        "közterület_jellege_22",
        "közterület_jellege_26",
        "épület_22",
        "épület_26",
        "lépcsőház_22",
        "lépcsőház_26"
    ], dropna=False)
    .size()
    .reset_index(name="darab")
    .sort_values("darab", ascending=False)
)

print(lh_osszesites.head(5).to_string(index=False))

print(
    "\nOlyan csoportok száma, ahol a darab = 1:",
    (lh_osszesites["darab"] == 1).sum()
)

mindketto_van_lh = lh_elteresek[
    lh_elteresek["lépcsőház_22"].notna()
    & lh_elteresek["lépcsőház_26"].notna()
].copy()

print("Mindkét évben van lépcsőházadat:", len(mindketto_van_lh))

print(mindketto_van_lh[[
    "település_azonosító",
    "közterület_neve_22",
    "közterület_jellege_22",
    "házszám_22",
    "épület_22",
    "épület_26",
    "lépcsőház_22",
    "lépcsőház_26"
]].to_string(index=False))

Lépcsőházeltérések, ahol az egyik adat NaN: 17
 település_azonosító közterület_neve_22 közterület_jellege_22 közterület_jellege_26 épület_22 épület_26 lépcsőház_22 lépcsőház_26  darab
                1011        Maléter Pál                  utca                  utca       NaN         C          NaN         0002      1
                1011   Povl Bang-Jensen                  utca                  utca       NaN         D          NaN         0002      1
                5075      Mátyás király                    út                    út       NaN       NaN         0002            A      1
                7013         Fáy András                  utca                  utca       NaN       NaN         0002            B      1
                9009       Dózsa György                  utca                 udvar       NaN       NaN         0001          NaN      1

Olyan csoportok száma, ahol a darab = 1: 19
Mindkét évben van lépcsőházadat: 9
 település_azonosító közterület_neve_22 közterület_